In [7]:
import vitaldb
import pandas as pd
import numpy as np
from scipy.stats import linregress
from tqdm.auto import tqdm

TRACKS = [
    "Solar8000/HR",
    "Solar8000/PLETH_SPO2",
    "Solar8000/ART_MBP"
]

INPUT_MINUTES = 15
PREDICTION_MINUTES = 30
STRIDE_MINUTES = 5

INPUT_SECONDS = INPUT_MINUTES * 60
PREDICTION_SECONDS = PREDICTION_MINUTES * 60
STRIDE_SECONDS = STRIDE_MINUTES * 60


def slope(series):
    series = series.dropna()

    if len(series) < 2:
        return np.nan

    return linregress(
        np.arange(len(series)),
        series.values
    ).slope


def generate_windows(df):

    windows = []

    total_len = len(df)

    start = 0

    while (
        start
        + INPUT_SECONDS
        + PREDICTION_SECONDS
        <= total_len
    ):

        input_df = df.iloc[
            start :
            start + INPUT_SECONDS
        ]

        prediction_df = df.iloc[
            start + INPUT_SECONDS :
            start + INPUT_SECONDS + PREDICTION_SECONDS
        ]

        windows.append(
            (input_df, prediction_df)
        )

        start += STRIDE_SECONDS

    return windows

In [8]:
def extract_trend_features(input_df):

    hr = input_df["Solar8000/HR"]
    spo2 = input_df["Solar8000/PLETH_SPO2"]
    map_ = input_df["Solar8000/ART_MBP"]

    return {

        # HR
        "hr_mean": hr.mean(),
        "hr_std": hr.std(),
        "hr_slope": slope(hr),

        # SpO2
        "spo2_mean": spo2.mean(),
        "spo2_std": spo2.std(),
        "spo2_slope": slope(spo2),

        # MAP
        "map_mean": map_.mean(),
        "map_std": map_.std(),
        "map_slope": slope(map_)
    }

In [9]:
def create_labels(prediction_df):

    hr = prediction_df["Solar8000/HR"]
    spo2 = prediction_df["Solar8000/PLETH_SPO2"]
    map_ = prediction_df["Solar8000/ART_MBP"]

    return {

        "tachycardia":
            int(hr.max() > 110),

        "hypotension":
            int(map_.min() < 65),

        "hypoxia":
            int(spo2.min() < 90)
    }

In [ ]:
def process_case(caseid):

    try:

        arr = vitaldb.load_case(
            caseid,
            TRACKS,
            interval=1
        )

        df = pd.DataFrame(
            arr,
            columns=TRACKS
        )

        # Skip useless cases
        if (
            df["Solar8000/ART_MBP"]
            .notna()
            .sum()
            < 900
        ):
            return []

        windows = generate_windows(df)

        rows = []

        for idx, (
            input_df,
            prediction_df
        ) in enumerate(windows):

            hr = input_df["Solar8000/HR"]
            spo2 = input_df["Solar8000/PLETH_SPO2"]
            map_ = input_df["Solar8000/ART_MBP"]

            # Skip poor-quality windows
            if (
                hr.notna().sum() < 300
                or spo2.notna().sum() < 300
                or map_.notna().sum() < 300
            ):
                continue

            feats = extract_trend_features(
                input_df
            )

            labels = create_labels(
                prediction_df
            )

            rows.append({

                "caseid": caseid,
                "window_id": idx,

                **feats,

                **labels
            })

        return rows

    except Exception:

        return []

In [13]:
import pandas as pd

df = pd.read_csv("./data/raw/window_dataset.csv")

caseids = sorted(df["caseid"].unique())

print(len(caseids))

3494


In [21]:
all_rows = []

for caseid in tqdm(caseids):

    rows = process_case(caseid)

    all_rows.extend(rows)

feature_df = pd.DataFrame(all_rows)

feature_df.to_csv(
    "window_features.csv",
    index=False
)

print(feature_df.shape)

100%|██████████| 3494/3494 [31:04<00:00,  1.87it/s] 


(124448, 14)


In [22]:
import pandas as pd

df = pd.read_csv("window_features.csv")

print(df.shape)
print(df.head())

print(df.isna().sum())

(124448, 14)
   caseid  window_id    hr_mean     hr_std  hr_slope   spo2_mean  spo2_std  \
0       1          0  93.284424  15.877655  0.073537   99.282852  1.254526   
1       1          1  95.339996  15.923326  0.020179   99.982224  0.162509   
2       1          2  92.995552  15.471073 -0.114539   99.997780  0.047140   
3       1          3  80.593330  10.291582 -0.069161   99.997780  0.047140   
4       1          4  72.591110   3.706602 -0.027258  100.000000  0.000000   

   spo2_slope   map_mean    map_std  map_slope  tachycardia  hypotension  \
0    0.007227  10.714922  41.783764   0.214405            1            1   
1    0.000162  46.115555  53.896751   0.348676            1            1   
2    0.000009  75.186668  37.560711   0.116285            1            1   
3    0.000029  81.922226  13.100119  -0.094773            1            1   
4    0.000000  70.199997   7.285461  -0.054307            1            1   

   hypoxia  
0        0  
1        0  
2        0  
3        

In [26]:
import pandas as pd

# Load
df = pd.read_csv("window_features.csv")

print("Original shape:", df.shape)

# =====================================
# Remove rows with missing features
# =====================================

df = df.dropna(
    subset=[
        "hr_mean",
        "hr_std",
        "hr_slope",
        "spo2_mean",
        "spo2_std",
        "spo2_slope",
        "map_mean",
        "map_std",
        "map_slope"
    ]
)

print("After NaN removal:", df.shape)

# =====================================
# Remove physiologically impossible values
# =====================================

df = df[
    (df["map_mean"] >= 30) &
    (df["map_mean"] <= 200) &

    (df["hr_mean"] >= 20) &
    (df["hr_mean"] <= 220) &

    (df["spo2_mean"] >= 50) &
    (df["spo2_mean"] <= 100)
]

print("After outlier removal:", df.shape)

# =====================================
# Check label balance
# =====================================

print("\nLabel rates:")
print("Tachycardia :", df["tachycardia"].mean())
print("Hypotension :", df["hypotension"].mean())
print("Hypoxia     :", df["hypoxia"].mean())

# =====================================
# Save cleaned dataset
# =====================================

df.to_csv(
    "window_features_clean.csv",
    index=False
)

print("\nSaved: window_features_clean.csv")

Original shape: (124448, 14)
After NaN removal: (120354, 14)
After outlier removal: (117010, 14)

Label rates:
Tachycardia : 0.2894880779420562
Hypotension : 0.5585078198444577
Hypoxia     : 0.08672763011708401

Saved: window_features_clean.csv


In [4]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

df = pd.read_csv("window_features_clean.csv")

FEATURES = [
    "hr_mean",
    "hr_std",
    "hr_slope",
    "spo2_mean",
    "spo2_std",
    "spo2_slope",
    "map_mean",
    "map_std",
    "map_slope"
]

groups = df["caseid"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(df, groups=groups)
)

train_df = df.iloc[train_idx]
test_df = df.iloc[test_idx]

print("Train:", train_df.shape)
print("Test :", test_df.shape)

print(
    "Train patients:",
    train_df["caseid"].nunique()
)

print(
    "Test patients:",
    test_df["caseid"].nunique()
)

Train: (93328, 14)
Test : (23682, 14)
Train patients: 2619
Test patients: 655


In [5]:
%pip install lightgbm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 171.8 kB/s eta 0:00:00m eta 0:00:010:01:02
Note: you may need to restart the kernel to use updated packages.


In [14]:
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score

X_train = train_df[FEATURES]
y_train = train_df["hypotension"]

X_test = test_df[FEATURES]
y_test = test_df["hypotension"]

model_h = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42
)

model_h.fit(X_train, y_train)

pred_prob = model_h.predict_proba(X_test)[:,1]

auc = roc_auc_score(
    y_test,
    pred_prob
)

print("Hypotension AUROC:", auc)

[LightGBM] [Info] Number of positive: 52316, number of negative: 41012
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000777 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2291
[LightGBM] [Info] Number of data points in the train set: 93328, number of used features: 9
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.560561 -> initscore=0.243438
[LightGBM] [Info] Start training from score 0.243438
Hypotension AUROC: 0.7219163900760786


In [15]:
import pandas as pd

importance = pd.DataFrame({
    "feature": FEATURES,
    "importance": model_h.feature_importances_
})

print(
    importance.sort_values(
        "importance",
        ascending=False
    )
)

      feature  importance
0     hr_mean        1368
6    map_mean        1326
7     map_std        1311
1      hr_std        1130
8   map_slope        1116
2    hr_slope         781
4    spo2_std         723
3   spo2_mean         706
5  spo2_slope         539


In [16]:
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score

X_train = train_df[FEATURES]
y_train = train_df["tachycardia"]

X_test = test_df[FEATURES]
y_test = test_df["tachycardia"]

model_t = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42
)

model_t.fit(X_train, y_train)

pred_prob = model_t.predict_proba(X_test)[:,1]

auc = roc_auc_score(
    y_test,
    pred_prob
)

print("Tachycardia AUROC:", auc)

[LightGBM] [Info] Number of positive: 27196, number of negative: 66132
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.023717 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2291
[LightGBM] [Info] Number of data points in the train set: 93328, number of used features: 9
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.291402 -> initscore=-0.888583
[LightGBM] [Info] Start training from score -0.888583
Tachycardia AUROC: 0.773118382179483


In [17]:
import pandas as pd

importance = pd.DataFrame({
    "feature": FEATURES,
    "importance": model_t.feature_importances_
})

print(
    importance.sort_values(
        "importance",
        ascending=False
    )
)

      feature  importance
0     hr_mean        1581
1      hr_std        1327
6    map_mean        1099
2    hr_slope        1094
7     map_std        1089
8   map_slope         804
3   spo2_mean         781
4    spo2_std         682
5  spo2_slope         543


In [21]:
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score

X_train = train_df[FEATURES]
y_train = train_df["hypoxia"]

X_test = test_df[FEATURES]
y_test = test_df["hypoxia"]

model_hy= LGBMClassifier(
    n_estimators=550,
    learning_rate=0.005,
    num_leaves=42,
    random_state=42
)

model_hy.fit(X_train, y_train)

pred_prob = model_hy.predict_proba(X_test)[:,1]

auc = roc_auc_score(
    y_test,
    pred_prob
)

print("Hypoxia AUROC:", auc)

[LightGBM] [Info] Number of positive: 7992, number of negative: 85336
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001363 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2291
[LightGBM] [Info] Number of data points in the train set: 93328, number of used features: 9
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.085633 -> initscore=-2.368155
[LightGBM] [Info] Start training from score -2.368155
Hypoxia AUROC: 0.6835639241633322
